# 📢 Prédiction du taux de clic (CTR) pour la publicité en ligne

**Objectif** : prédire la probabilité qu'un utilisateur clique sur une publicité, sur le jeu de données public de Criteo
(*Display Advertising Challenge*, ~45 millions d'affichages sur 7 jours).

Pourquoi c'est important : lors d'une enchère en temps réel, la valeur d'un affichage vaut `P(clic) × valeur d'un clic`.
Le modèle doit donc **bien classer** (AUC) **et** donner des **probabilités justes** (log loss, calibration).

Plan du notebook :
1. Chargement et découpage **chronologique** (on entraîne sur le passé, on teste sur le futur)
2. Préparation des variables (log des comptages, catégories rares, *hashing trick*)
3. Trois modèles : **régression logistique** (référence industrielle), **LightGBM**, **DeepFM** (PyTorch)
4. Évaluation : log loss, AUC, *Normalized Entropy*, calibration
5. (Optionnel) Passage à l'échelle avec **PySpark**

In [ ]:
# CONFIG : chemin des données et nombre de lignes à lire
DATA_PATH = "data/train.txt"   # fichier train.txt du jeu Criteo (voir data/README.md)
NROWS = 1_000_000              # 1 million de lignes : ~15 min sur Colab ; réduire si manque de RAM
RUN_SPARK = False              # passer à True pour la section PySpark (optionnelle)

### Récupérer les données (à lancer une seule fois, par exemple sur Google Colab)
Le fichier complet fait ~4 Go compressé. On ne garde que les premières lignes, qui sont les plus anciennes.

In [ ]:
# Décommenter pour télécharger les données (Colab / Linux)
# !mkdir -p data
# !wget -q --show-progress -O dac.tar.gz https://ndownloader.figshare.com/files/10082655
# !tar -xzOf dac.tar.gz train.txt | head -n 1000000 > data/train.txt
# !wc -l data/train.txt

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.feature_extraction import FeatureHasher
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
import lightgbm as lgb
import torch
from torch import nn

torch.manual_seed(0)
np.random.seed(0)
print("Librairies chargées. GPU disponible :", torch.cuda.is_available())

## 1. Chargement et découpage chronologique
Format : `label`, 13 entiers `I1..I13` (surtout des comptages), 26 catégories hachées `C1..C26`, séparés par des tabulations.
Les lignes sont **dans l'ordre du temps** : un découpage aléatoire « tricherait » en apprenant sur le futur.

In [ ]:
INT_COLS = [f"I{i}" for i in range(1, 14)]
CAT_COLS = [f"C{i}" for i in range(1, 27)]
COLS = ["label"] + INT_COLS + CAT_COLS

dtypes = {"label": "int8", **{c: "float32" for c in INT_COLS}, **{c: "string" for c in CAT_COLS}}
t0 = time.time()
df = pd.read_csv(DATA_PATH, sep="\t", header=None, names=COLS, dtype=dtypes, nrows=NROWS)
print(f"{len(df):,} lignes chargées en {time.time() - t0:.1f}s")

# 80 % entraînement (le plus ancien), 10 % validation, 10 % test (le plus récent)
n = len(df)
n_train, n_valid = int(0.8 * n), int(0.1 * n)
train = df.iloc[:n_train].reset_index(drop=True)
valid = df.iloc[n_train:n_train + n_valid].reset_index(drop=True)
test = df.iloc[n_train + n_valid:].reset_index(drop=True)
print(f"CTR moyen : train={train.label.mean():.4f}  valid={valid.label.mean():.4f}  test={test.label.mean():.4f}")
print("Valeurs manquantes (moyenne par colonne) :", f"{df.isna().mean().mean():.1%}")

> ⚠️ Le CTR de ce fichier est bien plus élevé que dans la réalité : Criteo a sous-échantillonné clics et non-clics
> à des taux différents. Pour utiliser le modèle en vrai, il faudrait recalibrer : `q = p / (p + (1 - p) / w)`
> où `w` est le taux de conservation des non-clics (He et al., 2014).

## 2. Préparation des variables (ajustée sur l'entraînement uniquement)
- **Entiers** : `log(1 + x)` car les comptages ont des valeurs extrêmes ; manquant → 0 + indicateur de manquant.
- **Catégories** : on garde les valeurs vues **au moins 10 fois** dans l'entraînement. Les valeurs rares ou jamais vues
  partagent l'identifiant 1 ; les manquantes ont l'identifiant 0.

In [ ]:
MIN_COUNT = 10
MISSING_ID, RARE_ID = 0, 1

def transform_int(s):
    v = s.to_numpy(dtype="float64", na_value=np.nan)
    out = np.where(v > 0, np.log1p(np.where(v > 0, v, 0.0)), v)
    return np.nan_to_num(out, nan=0.0).astype("float32")

# Vocabulaire des catégories, appris sur le train seulement (pas de fuite)
vocab = {}
for c in CAT_COLS:
    counts = train[c].value_counts(dropna=True)
    frequent = counts[counts >= MIN_COUNT].index
    vocab[c] = {val: i + 2 for i, val in enumerate(frequent)}
cardinalities = [len(vocab[c]) + 2 for c in CAT_COLS]

def encode(d):
    dense = np.stack([transform_int(d[c]) for c in INT_COLS], axis=1)
    missing = d[INT_COLS].isna().to_numpy(dtype="float32")
    dense = np.concatenate([dense, missing], axis=1)                 # 26 colonnes numériques
    cats = np.empty((len(d), len(CAT_COLS)), dtype="int64")
    for j, c in enumerate(CAT_COLS):
        ids = d[c].map(vocab[c])
        ids = ids.where(d[c].isna() | ids.notna(), RARE_ID)          # présente mais rare/inconnue -> 1
        cats[:, j] = ids.fillna(MISSING_ID).to_numpy(dtype="int64")   # manquante -> 0
    return dense, cats

enc = {name: encode(d) for name, d in [("train", train), ("valid", valid), ("test", test)]}
y = {name: d["label"].to_numpy() for name, d in [("train", train), ("valid", valid), ("test", test)]}
print("Nombre de valeurs fréquentes par catégorie (min / max) :", min(cardinalities), "/", max(cardinalities))

## 3. Métriques
- **Log loss** : juge les probabilités (métrique de production).
- **AUC** : juge le classement.
- **Normalized Entropy (NE)** : log loss du modèle / log loss d'un modèle qui prédit toujours le CTR moyen. NE < 1 = le modèle apporte de l'information.
- **Calibration** : CTR moyen prédit / CTR observé (idéal : 1).

In [ ]:
def evaluate(y_true, p):
    p = np.clip(np.asarray(p, dtype="float64"), 1e-7, 1 - 1e-7)
    base = np.clip(y_true.mean(), 1e-7, 1 - 1e-7)
    base_ll = -(base * np.log(base) + (1 - base) * np.log(1 - base))
    ll = log_loss(y_true, p)
    return {"logloss": ll, "AUC": roc_auc_score(y_true, p), "NE": ll / base_ll, "calibration": p.mean() / y_true.mean()}

results, preds = {}, {}

## 4. Modèle 1 : régression logistique avec *hashing trick*
Chaque couple `colonne=valeur` est envoyé par une fonction de hachage vers une des 2^20 positions d'un vecteur creux :
pas de dictionnaire à stocker, nouvelles valeurs gérées automatiquement. C'est le standard industriel historique.
Les entiers sont aussi découpés en paquets (log arrondi) pour capter des effets non linéaires.

In [ ]:
def hashed_features(d, n_features=2**20):
    tokens = [[] for _ in range(len(d))]
    for c in INT_COLS:
        v, na = transform_int(d[c]), d[c].isna().to_numpy()
        for i in range(len(d)):
            tokens[i].append(f"{c}=NA" if na[i] else f"{c}={int(round(v[i]))}")
    for c in CAT_COLS:
        vals = d[c].fillna("NA").to_numpy()
        for i in range(len(d)):
            tokens[i].append(f"{c}={vals[i]}")
    return FeatureHasher(n_features=n_features, input_type="string", alternate_sign=False).transform(tokens).tocsr()

t0 = time.time()
logreg = LogisticRegression(C=0.1, solver="liblinear", max_iter=200)
logreg.fit(hashed_features(train), y["train"])
preds["logreg"] = logreg.predict_proba(hashed_features(test))[:, 1]
results["logreg"] = {**evaluate(y["test"], preds["logreg"]), "temps (s)": round(time.time() - t0, 1)}
results["logreg"]

## 5. Modèle 2 : LightGBM
Gradient boosting : il apprend seul les non-linéarités et les interactions, et coupe directement sur les catégories.
Le nombre d'arbres est choisi par *early stopping* sur la log loss de validation.

In [ ]:
def to_frame(dense, cats):
    X = pd.DataFrame(dense, columns=INT_COLS + [f"{c}_missing" for c in INT_COLS])
    for j, c in enumerate(CAT_COLS):
        X[c] = pd.Categorical(cats[:, j])
    return X

t0 = time.time()
params = {"objective": "binary", "metric": "binary_logloss", "learning_rate": 0.05, "num_leaves": 63,
          "min_data_in_leaf": 100, "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1,
          "lambda_l2": 1.0, "cat_smooth": 10, "verbose": -1}
dtrain = lgb.Dataset(to_frame(*enc["train"]), y["train"], categorical_feature=CAT_COLS)
dvalid = lgb.Dataset(to_frame(*enc["valid"]), y["valid"], reference=dtrain)
booster = lgb.train(params, dtrain, num_boost_round=2000, valid_sets=[dvalid],
                    callbacks=[lgb.early_stopping(50, verbose=False)])
preds["lgbm"] = booster.predict(to_frame(*enc["test"]), num_iteration=booster.best_iteration)
results["lgbm"] = {**evaluate(y["test"], preds["lgbm"]), "temps (s)": round(time.time() - t0, 1)}
print("Nombre d'arbres retenus :", booster.best_iteration)
results["lgbm"]

## 6. Modèle 3 : DeepFM (PyTorch)
Chaque catégorie est représentée par un **embedding** (vecteur de dimension 16). Le logit est la somme de :
- une partie **linéaire** ;
- une **Factorization Machine** : toutes les interactions par paires `⟨v_i, v_j⟩`, calculées en O(n·k) grâce à
  `Σ_{i<j} ⟨v_i, v_j⟩ = ½ Σ_f [ (Σ_i v_if)² − Σ_i v_if² ]` ;
- un **MLP** (256 → 128) pour les interactions d'ordre supérieur.

Entraînement : Adam, entropie croisée binaire, *early stopping* sur la log loss de validation.

In [ ]:
class DeepFM(nn.Module):
    def __init__(self, cardinalities, n_dense, emb_dim=16, hidden=(256, 128), dropout=0.2):
        super().__init__()
        offsets = np.concatenate([[0], np.cumsum(cardinalities)[:-1]])
        self.register_buffer("offsets", torch.as_tensor(offsets, dtype=torch.long))
        total = int(sum(cardinalities))
        self.embedding = nn.Embedding(total, emb_dim)      # v_i (partagés par FM et MLP)
        self.linear_cat = nn.Embedding(total, 1)           # poids d'ordre 1
        self.linear_dense = nn.Linear(n_dense, 1)
        self.dense_proj = nn.Linear(n_dense, emb_dim)      # les numériques participent aussi aux interactions
        nn.init.normal_(self.embedding.weight, std=0.01)
        nn.init.zeros_(self.linear_cat.weight)
        layers, d_in = [], (len(cardinalities) + 1) * emb_dim + n_dense
        for h in hidden:
            layers += [nn.Linear(d_in, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            d_in = h
        self.mlp = nn.Sequential(*layers, nn.Linear(d_in, 1))

    def forward(self, dense, cats):
        idx = cats + self.offsets
        emb = torch.cat([self.embedding(idx), self.dense_proj(dense).unsqueeze(1)], dim=1)   # [B, F+1, k]
        linear = self.linear_cat(idx).sum(dim=(1, 2)) + self.linear_dense(dense).squeeze(1)
        fm = 0.5 * (emb.sum(dim=1).pow(2) - emb.pow(2).sum(dim=1)).sum(dim=1)
        deep = self.mlp(torch.cat([emb.flatten(1), dense], dim=1)).squeeze(1)
        return linear + fm + deep                                                            # logits

device = "cuda" if torch.cuda.is_available() else "cpu"
T = {k: (torch.tensor(enc[k][0]), torch.tensor(enc[k][1]), torch.tensor(y[k], dtype=torch.float32)) for k in enc}

@torch.no_grad()
def predict_logits(model, dense, cats, bs=8192):
    model.eval()
    return torch.cat([model(dense[i:i + bs].to(device), cats[i:i + bs].to(device)).cpu() for i in range(0, len(dense), bs)])

t0 = time.time()
model = DeepFM(cardinalities, n_dense=T["train"][0].shape[1]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
loss_fn = nn.BCEWithLogitsLoss()
best, best_state, bad, BS = float("inf"), None, 0, 4096
for epoch in range(10):
    model.train()
    perm = torch.randperm(len(T["train"][2]))
    for i in range(0, len(perm), BS):
        b = perm[i:i + BS]
        opt.zero_grad()
        loss = loss_fn(model(T["train"][0][b].to(device), T["train"][1][b].to(device)), T["train"][2][b].to(device))
        loss.backward()
        opt.step()
    val_loss = float(loss_fn(predict_logits(model, *T["valid"][:2]), T["valid"][2]))
    print(f"époque {epoch + 1} : log loss validation = {val_loss:.5f}")
    if val_loss < best - 1e-5:
        best, bad, best_state = val_loss, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= 2:      # early stopping
            break
model.load_state_dict(best_state)
preds["deepfm"] = torch.sigmoid(predict_logits(model, *T["test"][:2])).numpy()
results["deepfm"] = {**evaluate(y["test"], preds["deepfm"]), "temps (s)": round(time.time() - t0, 1)}
results["deepfm"]

## 7. Comparaison des trois modèles

In [ ]:
table = pd.DataFrame(results).T.round(4)
print(f"{len(df):,} lignes, découpage chronologique 80/10/10, CTR moyen du test = {y['test'].mean():.4f}")
table

In [ ]:
# Courbe de calibration : CTR prédit vs CTR observé, par déciles de prédiction
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, p in preds.items():
    order = np.argsort(p)
    bins = np.array_split(order, 10)
    ax.plot([p[b].mean() for b in bins], [y["test"][b].mean() for b in bins], "o-", label=name)
lim = [0, max(max(p.max() for p in preds.values()), y["test"].mean() * 2)]
ax.plot(lim, lim, "k--", lw=1, label="calibration parfaite")
ax.set_xlabel("CTR moyen prédit (par décile)")
ax.set_ylabel("CTR observé")
ax.set_title("Calibration des modèles (jeu de test)")
ax.legend()
plt.show()

### Lecture des résultats (à compléter avec tes chiffres)
- Quel modèle a la **meilleure log loss** ? la **meilleure AUC** ? Sont-ils les mêmes ?
- Tous ont-ils **NE < 1** ? De combien chaque modèle améliore-t-il la référence « CTR moyen » ?
- La **calibration** est-elle proche de 1 ? La courbe suit-elle la diagonale ?
- Le gain de DeepFM vaut-il son coût (temps d'entraînement, complexité) à cette échelle ?

## 8. (Optionnel) Passage à l'échelle avec PySpark
Le fichier complet (~11 Go) ne tient pas dans Pandas sur un portable. Spark calcule les fréquences des 26 catégories
en une seule agrégation distribuée (format long `(colonne, valeur)` avec `explode`, puis `groupBy` + `count`).

In [ ]:
if RUN_SPARK:
    from pyspark.sql import SparkSession, functions as F
    spark = SparkSession.builder.appName("criteo-vocab").getOrCreate()
    sdf = spark.read.option("sep", "\t").csv(DATA_PATH, header=False).toDF(*COLS)
    long = sdf.select(F.explode(F.array(*[F.struct(F.lit(c).alias("col"), F.col(c).alias("val")) for c in CAT_COLS])).alias("kv"))
    vocab_spark = (long.select("kv.col", "kv.val").where(F.col("val").isNotNull())
                   .groupBy("col", "val").count().where(F.col("count") >= MIN_COUNT))
    vocab_spark.groupBy("col").count().orderBy("col").show(26)
    spark.stop()
else:
    print("Section PySpark ignorée (RUN_SPARK = False).")

## Références
- He et al., *Practical Lessons from Predicting Clicks on Ads at Facebook*, 2014 (Normalized Entropy, recalibration).
- Rendle, *Factorization Machines*, 2010.
- Guo et al., *DeepFM: A Factorization-Machine based Neural Network for CTR Prediction*, 2017.